# Fitness and Nutrition Project: Predicting Actual Body Weight

This notebook is written for our report topic. The goal is to predict a person’s **actual body weight** using fitness, exercise, and nutrition-related variables. The workflow includes data cleaning, exploratory data analysis, model training, model comparison, prediction visualization, and feature importance.

In [1]:
# Student information
NAME = "Melvin Lopez"
PROJECT_TOPIC = "Predicting Actual Body Weight from Fitness and Nutrition Data"

In [2]:
# 1. Import libraries
import re
from pathlib import Path
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

output_dir = "figures"
os.makedirs(output_dir, exist_ok=True)
print(os.getcwd())
print(os.listdir())
def save_fig(name):
    path = os.path.join(output_dir, name)
    plt.savefig(path, dpi=300, bbox_inches="tight")

pd.set_option("display.max_columns", None)

/home/mnl90/assignments
['.ipynb_checkpoints', 'lab3', 'Calorie&ExceriseBurnPrediction_mnl90-Copy1.ipynb', 'lab4', 'FINAL.ipynb', 'Lab1', 'body_weight_prediction_report_version.ipynb', 'body_weight_prediction_report_version (2).ipynb', 'Lab2', 'figures', 'README.txt', 'Calorie&ExceriseBurnPrediction_mnl90.ipynb', 'exercise_dataset_DIRTY.csv']


In [3]:
# 2. Load the dataset
# Use the dirty practice file if it is available. Otherwise, use the original dataset.
possible_files = [
    "exercise_dataset_DIRTY.csv",
    "exercise_dataset.csv"
]

data_file = None
for file_name in possible_files:
    if Path(file_name).exists():
        data_file = file_name
        break

if data_file is None:
    raise FileNotFoundError("Put exercise_dataset.csv or exercise_dataset_DIRTY.csv in the same folder as this notebook.")

raw_df = pd.read_csv(data_file)
print(f"Loaded file: {data_file}")
print(f"Rows: {raw_df.shape[0]}, Columns: {raw_df.shape[1]}")
raw_df.head()

Loaded file: exercise_dataset_DIRTY.csv
Rows: 3960, Columns: 13


,ID,Exercise,Calories Burn,Dream Weight,Actual Weight,Age,Gender,Duration,Heart Rate,BMI,Weather Conditions,Exercise Intensity,Notes
0,1,Exercise 2,286.9598505,91.89253067,96.30111546,45,Male,37,170,29.42627467,Rainy,5,felt good
1,2,Workout 7,343.4530361,64.16509681,61.1046681,25,male,43,142,21.28634599,rainy,5,tired
2,3,Exercise 4,NaN,70.84622352,71.76672384,20,Male,20,148,27.8995916,Cloudy,4,felt good
3,4,Exercise 5,127.1838584,79.47700756,82.98445557,33,Male,39,170,33.72955245,Sunny,10,low energy
4,5,Exercise 10,416.3183735,89.96022608,85.64317443,29,Female,34,118,23.28611341,Cloudy,3,bad sleep


In [4]:
# 3. Clean column names and messy values
# This section is important because the dirty version may contain extra spaces,
# inconsistent text labels, missing values, and numbers stored as strings.

df = raw_df.copy()
df.columns = df.columns.str.strip()

missing_tokens = {"", " ", "na", "n/a", "none", "null", "missing", "?", "--"}

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace({token: np.nan for token in missing_tokens})

# Helper function to pull numeric values out of messy strings like "45 years" or "180 bpm".
def clean_numeric(series):
    return (
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.extract(r"(-?\d+\.?\d*)", expand=False)
        .astype(float)
    )

numeric_cols = [
    "Calories Burn", "Dream Weight", "Actual Weight", "Age",
    "Duration", "Heart Rate", "BMI", "Exercise Intensity"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = clean_numeric(df[col])

# Standardize categorical columns.
if "Gender" in df.columns:
    df["Gender"] = df["Gender"].str.lower().str.strip()
    df["Gender"] = df["Gender"].replace({"m": "male", "f": "female"})
    df["Gender"] = df["Gender"].str.title()

if "Weather Conditions" in df.columns:
    df["Weather Conditions"] = df["Weather Conditions"].str.lower().str.strip().str.title()

if "Exercise" in df.columns:
    df["Exercise"] = df["Exercise"].str.lower().str.strip()
    df["Exercise"] = df["Exercise"].str.replace("workout", "exercise", regex=False)
    df["Exercise"] = df["Exercise"].str.title()

# Remove exact duplicate rows.
duplicate_count = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)

print(f"Duplicate rows removed: {duplicate_count}")
print("Missing values after basic cleaning:")
df.isna().sum()

Duplicate rows removed: 51
Missing values after basic cleaning:


ID                     0
Exercise               0
Calories Burn         35
Dream Weight          32
Actual Weight         26
Age                   30
Gender                 7
Duration              37
Heart Rate            27
BMI                   32
Weather Conditions     3
Exercise Intensity    36
Notes                  0
dtype: int64

In [5]:
# 4. Choose the target variable and remove unrealistic target values
# Our report focuses on predicting actual body weight.

target = "Actual Weight"

if target not in df.columns:
    raise ValueError(f"Target column '{target}' was not found in the dataset.")

# Drop rows where the target is missing.
df = df.dropna(subset=[target]).copy()

# Remove impossible or extreme body-weight entries.
# The dataset appears to store weight in kilograms, so this range keeps realistic adult values.
df = df[(df[target] > 30) & (df[target] < 250)].copy()

print(f"Rows available after target cleaning: {len(df)}")
df[[target]].describe()

Rows available after target cleaning: 3883


,Actual Weight
count,3883.000000
mean,75.636553
std,16.019639
min,45.783747
25%,62.558748
50%,75.843238
75%,88.264634
max,216.600000


In [ ]:
# 5. Exploratory Data Analysis
# These graphs match the report sections where we discuss distributions and relationships.

plt.figure(figsize=(8, 5))
plt.hist(df["Actual Weight"].dropna(), bins=30,edgecolor="black")
plt.xlim(40, 140)
plt.xlabel("Actual Weight")
plt.ylabel("Number of Participants")
plt.title("Distribution of Actual Body Weight")
plt.tight_layout()
save_fig("weight_distribution.png")
plt.show()

plt.figure(figsize=(8, 5))
plt.scatter(df["BMI"], df["Actual Weight"], alpha=0.35)
plt.xlim(10, 50)
plt.ylim(40, 140)
plt.xlabel("BMI")
plt.ylabel("Actual Weight")
plt.title("BMI vs Actual Body Weight")
plt.grid(True, alpha=0.3)
plt.tight_layout()
save_fig("bmi_vs_weight.png")
plt.show()

plt.figure(figsize=(8, 5))
plt.scatter(df["Dream Weight"], df["Actual Weight"], alpha=0.35)
plt.xlim(40, 140)
plt.ylim(40, 140)
plt.xlabel("Dream Weight")
plt.ylabel("Actual Weight")
plt.title("Dream Weight vs Actual Body Weight")
plt.grid(True, alpha=0.3)
plt.tight_layout()
save_fig("dream_weight_vs_weight.png")
plt.show()

plt.figure(figsize=(8, 5))
plt.scatter(df["Duration"], df["Calories Burn"], alpha=0.35)
plt.xlim(0, 120)
plt.ylim(0, 1000)
plt.xlabel("Workout Duration")
plt.ylabel("Calories Burn")
plt.title("Workout Duration vs Calories Burn")
plt.grid(True, alpha=0.3)
plt.tight_layout()
save_fig("duration_vs_calories.png")
plt.show()

In [7]:
# 6. Prepare features for modeling
# We remove ID because it is only an identifier, and Notes because it is messy free-text.
# We also remove the target from the feature set so the model cannot cheat.

columns_to_drop = [target]
for col in ["ID", "Notes"]:
    if col in df.columns:
        columns_to_drop.append(col)

X = df.drop(columns=columns_to_drop)
y = df[target]

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Target variable:", target)
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=23
)

print(f"Training rows: {X_train.shape[0]}")
print(f"Testing rows: {X_test.shape[0]}")

Target variable: Actual Weight
Numeric features: ['Calories Burn', 'Dream Weight', 'Age', 'Duration', 'Heart Rate', 'BMI', 'Exercise Intensity']
Categorical features: ['Exercise', 'Gender', 'Weather Conditions']
Training rows: 3106
Testing rows: 777


In [8]:
# 7. Build preprocessing pipeline
# Numerical columns: fill missing values with the median and scale them.
# Categorical columns: fill missing values with the most common value and one-hot encode them.

numeric_pipeline = Pipeline(steps=[
    ("fill_missing_numbers", SimpleImputer(strategy="median")),
    ("scale_numbers", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("fill_missing_categories", SimpleImputer(strategy="most_frequent")),
    ("encode_categories", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

In [9]:
# 8. Train multiple regression models
# This follows the report: Linear Regression, Decision Tree Regressor, and Random Forest Regressor.

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree Regressor": DecisionTreeRegressor(max_depth=7, random_state=23),
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=200,
        max_depth=12,
        random_state=23,
        n_jobs=-1
    )
}

trained_models = {}
results = []

for model_name, estimator in models.items():
    model_pipe = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", estimator)
    ])
    
    model_pipe.fit(X_train, y_train)
    predictions = model_pipe.predict(X_test)
    
    mae = mean_absolute_error(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, predictions)
    
    trained_models[model_name] = model_pipe
    results.append({
        "Model": model_name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    })

results_df = pd.DataFrame(results).sort_values(by="RMSE")
results_df

,Model,MAE,MSE,RMSE,R2
2,Random Forest Regressor,3.479053,74.615756,8.638041,0.721665
1,Decision Tree Regressor,3.636091,91.482540,9.564651,0.658747
0,Linear Regression,4.583299,94.213964,9.706388,0.648558


In [ ]:
# 9. Pick the best model
# Lower RMSE is better because it means the predicted weight is closer to the actual weight.

best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]
best_predictions = best_model.predict(X_test)

print(f"Best model based on RMSE: {best_model_name}")
print(results_df.to_string(index=False))

In [ ]:
# 10. Actual vs predicted body weight visualization
# Points closer to the diagonal line represent stronger predictions.

plt.figure(figsize=(7, 7))
plt.scatter(y_test, best_predictions, alpha=0.45)
lowest_value = 40
highest_value = 140
plt.plot([lowest_value, highest_value],[lowest_value, highest_value], 
         linestyle="--")
plt.xlim(lowest_value, highest_value)
plt.ylim(lowest_value, highest_value)
plt.xlabel("Actual Body Weight")
plt.ylabel("Predicted Body Weight")
plt.title(f"Actual vs Predicted Body Weight ({best_model_name})")
plt.grid(True, alpha=0.3)
plt.tight_layout()
save_fig("actual_vs_predicted.png")
plt.show()

In [ ]:
# 11. Feature importance from the Random Forest model
# This connects to the report section where we explain which variables mattered most.

forest_pipeline = trained_models["Random Forest Regressor"]
fitted_preprocess = forest_pipeline.named_steps["preprocess"]
forest_model = forest_pipeline.named_steps["model"]

encoded_cat_names = fitted_preprocess.named_transformers_["categorical"]     .named_steps["encode_categories"]     .get_feature_names_out(categorical_features)

feature_names = np.array(numeric_features + list(encoded_cat_names))
importance_values = forest_model.feature_importances_

importance_df = (
    pd.DataFrame({"Feature": feature_names, "Importance": importance_values})
    .sort_values(by="Importance", ascending=False)
    .head(12)
)

importance_df

In [ ]:
# 12. Plot top feature importance values

top_features = importance_df.head(10)

plt.figure(figsize=(9, 6))

plt.barh(top_features["Feature"][::-1], top_features["Importance"][::-1])

plt.xlabel("Importance Score")
plt.title("Top Features for Predicting Actual Body Weight")

plt.tight_layout()
save_fig("feature_importance.png")
plt.show()

## Notes for the report

Use the model comparison table from `results_df` in the Evaluation section. Use the actual vs. predicted graph for the Prediction Visualization section. Use the feature importance table and graph to explain which variables were most helpful for predicting actual body weight.